#High Performance Computing - CIIC 5019
##Final Project - GPU-Accelerated-Network-Flow-Feature-Extraction

Dr. Wilson Rivera Gallego

## Loading the Dataset
1. Upload the dataset to Google Drive
2. Mount the drive to Google Colab
3. Insert the path for the dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Check the dataset is loaded. Then, replace the path in the macro for both the GPU and the CPU

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/dataset"

 6to4.pcap
 abis-accept-network.pcap
 abis-reject-network.pcap
 airtunes-1.pcap
 alp-sample1.pcap
 alp-sample2.pcap
 anony-tcp-std.pcap
 Apple_IP-over-IEEE_1394_Packet.pcap
 bfd-raw-auth-md5.pcap
 bfd-raw-auth-sha1.pcap
 bfd-raw-auth-simple.pcap
 c1222_std_example8.pcap
 chargen-tcp.pcap
 chargen-udp.pcap
 cisco-nexus10-erspan-marker.pcap
 cisco-nexus92-erspan-marker.pcap
 cmp_IR_sequence_OpenSSL-Cryptlib.pcap
 cmp_IR_sequence_-OpenSSL-EJBCA.pcap
 coap-cbor.pcap
 couchbase-lww.pcap
 couchbase_subdoc_multi.pcap
 dhcp-nanosecond.pcap
 DICOM_C-ECHO-echoscu.pcap
 dlep.pcap
 dnp3_read.pcap
 dnp3_select_operate.pcap
 dnp3_write.pcap
 dns_port.pcap
 dvb-ci_1.pcap
 dvb-ci_2.pcap
 dvb-ci_UV1_0000.pcap
 eapol-mka.pcap
 eia709.1-over-eia852.pcap
 eigrp-for-ipv6-auth.pcap
 eigrp-for-ipv6-updates.pcap
 eigrp-ipx.pcap
 exablaze_trailer.pcap
 fix.pcap
 fix-ssl.pcap
 gearman.pcap
 gopher.pcap
 greb_filterlist.pcap
 gsm-r.uus1.pcap
 hart_ip.pcap
 hdcp_authentication_sample.pcap
 homeplug_network_statis

# GPU Accelerated

## Pipeline Anatomy
```
[ Raw PCAP Files ]
       │ (Load & Batch via MPI)
       ▼
[ Packet Parsing Kernel (CUDA) ]
       │ Extract 5-tuple: (Source IP, Dest IP, Source Port, Dest Port, Protocol)
       ▼
[ Parallel Hashing & Grouping ]
       │ Resolve collisions & accumulate counts/bytes in GPU Global Memory
       ▼
[ Feature Reduction & Computation ]
       │ Calculate inter-arrival times, entropy, and flags
       ▼
[ Flow Records Output ]

In [24]:
%%file gpu.cu
#include <dirent.h>
#include <mpi.h>
#include <stdint.h>
#include <stdio.h>
#include <string.h>
#include <strings.h>
#include <sys/stat.h>

// DO NOT finish your path with "/"
#define DATASET_PATH "/content/drive/MyDrive/Colab Notebooks/dataset"

// ethernet types
#define IPv4 0x0800
#define ARP 0x0806
#define IPv6 0x86DD
#define VLAN_TAG 0x8100
#define QINQ_TAG 0x88A8
#define PPPoE_DISCOV 0x8863
#define PPPoE_SESSION 0x8864
#define MLPS_UNI 0x8847
#define MLPS_MULTI 0x8848

// IP protocol types
#define ICMP 1
#define TCP 6
#define UDP 17
#define SCTP 132

// arp operations
#define ARP_REQ 1
#define ARP_REP 2
// Hashing
#define HASH_TABLE_SIZE (1 << 20)
#define HASH_MASK (HASH_TABLE_SIZE - 1)
#define SLOT_EMPTY   0
#define SLOT_WRITING 1
#define SLOT_READY   2

/*
* Raw PCAP parsing -> pcap_parse.c
*/

// PCAP global header
typedef struct {
  uint32_t magic_number;   // number identifying PCAP format
  uint16_t version_major;  // Major version number
  uint16_t version_minor;  // Minor version number
  int32_t thiszone;        // GMT to local correction
  uint32_t sigfigs;        // Accuracy of timestamps
  uint32_t snaplen;        // Max length of captured packets
  uint32_t network;        // Data link type
} __attribute__((packed)) pcap_hdr_t;

// PCAP packet header
typedef struct {
  uint32_t ts_sec;    // Timestamp seconds
  uint32_t ts_usec;   // Timestamp microseconds (nanoseconds for 0xa1b23c4d)
  uint32_t incl_len;  // Number of bytes of packet saved in file
  uint32_t orig_len;  // Actual length of packet on the wire
} __attribute__((packed)) pcap_pkt_hdr_t;

// Per-packet metadata, one 32-byte record per packet (shared by host + GPU).
typedef struct {
  uint64_t offset;    // byte offset of the packet inside packet_data
  uint64_t ts_ns;     // capture timestamp, nanoseconds
  uint32_t caplen;    // bytes actually stored in packet_data (<= wire_len)
  uint32_t wire_len;  // length of the frame on the wire
  uint32_t file_id;   // index of the source pcap (keeps captures separate)
  uint32_t _pad;
} pkt_info_t;

// Set to 0 to stop printing one line per loaded file.
#define LOG_EACH_FILE 1

// A single captured packet larger than this means a corrupt record.
#define MAX_PKT_BYTES (1u << 20)

pkt_info_t* packets = NULL;
uint8_t* packet_data = NULL;

size_t total_data_bytes = 0;
size_t total_packets = 0;
size_t data_capacity = 0;    // Current allocated capacity for bytes
size_t packet_capacity = 0;  // Current allocated capacity for packets

static uint32_t Swap32(uint32_t v) { return __builtin_bswap32(v); }

// Grow both buffers geometrically. Returns 0 on success, -1 if out of memory.
static int EnsureCapacity(size_t need_packets, size_t need_bytes) {
  if (packets == NULL || need_packets > packet_capacity) {
    size_t cap = packet_capacity ? packet_capacity : 1024;
    while (cap < need_packets) cap *= 2;
    pkt_info_t* p = (pkt_info_t*)realloc(packets, cap * sizeof(pkt_info_t));
    if (p == NULL) return -1;
    packets = p;
    packet_capacity = cap;
  }
  if (packet_data == NULL || need_bytes > data_capacity) {
    size_t cap = data_capacity ? data_capacity : 65536;
    while (cap < need_bytes) cap *= 2;
    uint8_t* p = (uint8_t*)realloc(packet_data, cap);
    if (p == NULL) return -1;
    packet_data = p;
    data_capacity = cap;
  }
  return 0;
}

int PcapRead(const char* pcap_file, uint32_t file_id) {
  FILE* fd = fopen(pcap_file, "rb");
  if (fd == NULL) {
    fprintf(stderr, "Cannot open %s: ", pcap_file);
    perror(NULL);
    return 1;
  }

  pcap_hdr_t glob_hdr;
  if (fread(&glob_hdr, sizeof(glob_hdr), 1, fd) != 1) {
    fprintf(stderr, "%s: cannot read the pcap global header, skipped\n",
            pcap_file);
    fclose(fd);
    return 1;
  }

  // Classic pcap only. The magic tells us byte order and timestamp resolution.
  int swap = 0, nano = 0;
  switch (glob_hdr.magic_number) {
    case 0xa1b2c3d4: swap = 0; nano = 0; break;  // native, microseconds
    case 0xa1b23c4d: swap = 0; nano = 1; break;  // native, nanoseconds
    case 0xd4c3b2a1: swap = 1; nano = 0; break;  // swapped, microseconds
    case 0x4d3cb2a1: swap = 1; nano = 1; break;  // swapped, nanoseconds
    default:
      fprintf(stderr,
              "%s: unsupported format (magic 0x%08x, pcapng?), skipped\n",
              pcap_file, glob_hdr.magic_number);
      fclose(fd);
      return 1;
  }

  // The parser only understands Ethernet framing (linktype 1). The top 4 bits
  // of the field carry FCS info, so mask them off.
  uint32_t linktype =
      (swap ? Swap32(glob_hdr.network) : glob_hdr.network) & 0x0FFFFFFFu;
  if (linktype != 1) {
    fprintf(stderr, "%s: link type %u is not Ethernet, skipped\n", pcap_file,
            linktype);
    fclose(fd);
    return 1;
  }

  pcap_pkt_hdr_t pkt_hdr;
  while (fread(&pkt_hdr, sizeof(pkt_hdr), 1, fd) == 1) {
    uint32_t ts_sec = pkt_hdr.ts_sec;
    uint32_t ts_frac = pkt_hdr.ts_usec;
    uint32_t caplen = pkt_hdr.incl_len;
    uint32_t wirelen = pkt_hdr.orig_len;
    if (swap) {
      ts_sec = Swap32(ts_sec);
      ts_frac = Swap32(ts_frac);
      caplen = Swap32(caplen);
      wirelen = Swap32(wirelen);
    }

    if (caplen > MAX_PKT_BYTES) {
      fprintf(stderr, "%s: corrupt record (incl_len=%u), rest of file ignored\n",
              pcap_file, caplen);
      break;
    }

    if (EnsureCapacity(total_packets + 1, total_data_bytes + caplen) != 0) {
      fprintf(stderr, "Out of memory while loading %s\n", pcap_file);
      fclose(fd);
      return -1;
    }

    // Read the payload first; a cut-off last packet is dropped, not recorded.
    if (caplen > 0 &&
        fread(packet_data + total_data_bytes, 1, caplen, fd) != caplen) {
      fprintf(stderr, "%s: truncated final packet dropped\n", pcap_file);
      break;
    }

    pkt_info_t* p = &packets[total_packets];
    p->offset = total_data_bytes;
    p->ts_ns = (uint64_t)ts_sec * 1000000000ULL +
               (nano ? (uint64_t)ts_frac : (uint64_t)ts_frac * 1000ULL);
    p->caplen = caplen;
    p->wire_len = wirelen < caplen ? caplen : wirelen;  // guard bad orig_len
    p->file_id = file_id;
    p->_pad = 0;

    total_data_bytes += caplen;
    total_packets++;
  }

  fclose(fd);
  return 0;
}

static int HasSuffix(const char* s, const char* suffix) {
  size_t n = strlen(s), m = strlen(suffix);
  return n >= m && strcasecmp(s + n - m, suffix) == 0;
}

int DatasetRead(int my_rank, int num_procs) {
  struct dirent** list = NULL;
  // scandir + alphasort gives every rank the same file order, so the
  // round-robin split below never skips or duplicates a file.
  int n = scandir(DATASET_PATH, &list, NULL, alphasort);
  if (n < 0) {
    perror("Error opening dataset directory");
    return 1;
  }

  char filepath[1024];
  uint32_t file_id = 0;  // counts candidate files, identical on every rank
  int loaded = 0, skipped = 0, rc = 0;

  for (int i = 0; i < n && rc == 0; i++) {
    const char* name = list[i]->d_name;
    if (!HasSuffix(name, ".pcap") && !HasSuffix(name, ".cap")) continue;

    int len = snprintf(filepath, sizeof(filepath), "%s/%s", DATASET_PATH, name);
    if (len < 0 || (size_t)len >= sizeof(filepath)) continue;

    // d_type can be DT_UNKNOWN on some filesystems, so ask stat() instead
    struct stat st;
    if (stat(filepath, &st) != 0 || !S_ISREG(st.st_mode)) continue;

    uint32_t this_id = file_id++;
    if ((int)(this_id % (uint32_t)num_procs) != my_rank) continue;

    size_t before = total_packets;
    int r = PcapRead(filepath, this_id);
    if (r < 0) {
      rc = 1;
    } else if (r > 0) {
      skipped++;
    } else {
      loaded++;
#if LOG_EACH_FILE
      printf("[Rank %d] file_id=%u %s (%zu packets)\n", my_rank, this_id, name,
             total_packets - before);
#endif
    }
  }

  for (int i = 0; i < n; i++) free(list[i]);
  free(list);

  printf("[Rank %d] files loaded: %d, skipped: %d\n", my_rank, loaded, skipped);
  return rc;
}

// ---------------------------------------------------------------------------
/*
* GPU kernels for processing
*/
// ---------------------------------------------------------------------------

typedef struct {
  uint32_t file_id;
  uint32_t src_ip[4];
  uint32_t dst_ip[4];
  uint16_t src_port;  // 0 unless TCP/UDP/SCTP
  uint16_t dst_port;
  uint8_t proto;   // final upper-layer protocol (after IPv6 ext. headers)
  uint8_t ip_ver;  // 4 or 6
} flow_key_t;

typedef struct __align__(8) {
  uint32_t state;
  flow_key_t key;  // written once by the slot owner
  // aggregates (atomics)
  uint32_t packet_count;
  uint32_t min_pkt_len;
  uint32_t max_pkt_len;
  uint64_t byte_count;
  uint64_t first_seen_ts;  // ns
  uint64_t last_seen_ts;   // ns
}
flow_record_t;

typedef struct {
  unsigned long long non_flow;    // ARP, PPPoE discovery, unknown ethertype...
  unsigned long long malformed;   // truncated or invalid headers
  unsigned long long table_full;  // no free slot in the hash table
} parse_stats_t;

__device__ __forceinline__ uint16_t be16(const uint8_t* p) {
  return ((uint16_t)p[0] << 8) | p[1];
}

__device__ __forceinline__ uint32_t be32(const uint8_t* p) {
  return ((uint32_t)p[0] << 24) | ((uint32_t)p[1] << 16) |
         ((uint32_t)p[2] << 8) | (uint32_t)p[3];
}

__device__ __forceinline__ uint32_t Mix(uint32_t h, uint32_t x) {
  h ^= x;
  return h * 16777619u;  // FNV prime
}

__device__ uint32_t HashKey(const flow_key_t& k) {
  uint32_t h = 2166136261u;  // FNV offset basis
  h = Mix(h, k.file_id);
#pragma unroll
  for (int i = 0; i < 4; i++) h = Mix(h, k.src_ip[i]);
#pragma unroll
  for (int i = 0; i < 4; i++) h = Mix(h, k.dst_ip[i]);
  h = Mix(h, ((uint32_t)k.src_port << 16) | k.dst_port);
  h = Mix(h, ((uint32_t)k.proto << 8) | k.ip_ver);
  // murmur3 finalizer: a multiply only carries information upward, so without
  // this the bits kept by HASH_MASK ignore most of the high input bits (e.g.
  // the upper bits of the source port).
  h ^= h >> 16;
  h *= 0x85ebca6bu;
  h ^= h >> 13;
  h *= 0xc2b2ae35u;
  h ^= h >> 16;
  return h;
}

__device__ __forceinline__ void StoreKey(volatile flow_key_t* d,
                                         const flow_key_t& k) {
  d->file_id = k.file_id;
#pragma unroll
  for (int i = 0; i < 4; i++) {
    d->src_ip[i] = k.src_ip[i];
    d->dst_ip[i] = k.dst_ip[i];
  }
  d->src_port = k.src_port;
  d->dst_port = k.dst_port;
  d->proto = k.proto;
  d->ip_ver = k.ip_ver;
}

__device__ __forceinline__ void LoadKey(const volatile flow_key_t* s,
                                        flow_key_t* k) {
  k->file_id = s->file_id;
#pragma unroll
  for (int i = 0; i < 4; i++) {
    k->src_ip[i] = s->src_ip[i];
    k->dst_ip[i] = s->dst_ip[i];
  }
  k->src_port = s->src_port;
  k->dst_port = s->dst_port;
  k->proto = s->proto;
  k->ip_ver = s->ip_ver;
}

__device__ __forceinline__ bool KeyEq(const flow_key_t& a,
                                      const flow_key_t& b) {
  if (a.file_id != b.file_id || a.src_port != b.src_port ||
      a.dst_port != b.dst_port || a.proto != b.proto || a.ip_ver != b.ip_ver)
    return false;
#pragma unroll
  for (int i = 0; i < 4; i++) {
    if (a.src_ip[i] != b.src_ip[i] || a.dst_ip[i] != b.dst_ip[i]) return false;
  }
  return true;
}

__device__ bool AggregateFlow(flow_record_t* table, const flow_key_t& key,
                              uint32_t pkt_len, uint64_t ts) {
  uint32_t idx = HashKey(key) & HASH_MASK;

  for (uint32_t probes = 0; probes < HASH_TABLE_SIZE; probes++) {
    flow_record_t* slot = &table[idx];
    volatile flow_record_t* vslot = slot;

    // Try to claim an empty slot
    uint32_t prev = atomicCAS(&slot->state, SLOT_EMPTY, SLOT_WRITING);

    if (prev == SLOT_EMPTY) {
      // We own it: write the key, then publish
      StoreKey(&vslot->key, key);
      // table is zero-initialised; min/first need a "maximum" starting value
      vslot->min_pkt_len = UINT32_MAX;
      vslot->first_seen_ts = UINT64_MAX;
      __threadfence();  // key visible before READY
      atomicExch(&slot->state, SLOT_READY);
    } else {
      // Someone else owns it: wait until their key is fully published
      while (vslot->state != SLOT_READY) {
      }
      __threadfence();
    }

    // Slot is READY: does the key match?
    flow_key_t cur;
    LoadKey(&vslot->key, &cur);
    if (KeyEq(cur, key)) {
      atomicAdd(&slot->packet_count, 1);
      atomicAdd((unsigned long long*)&slot->byte_count,
                (unsigned long long)pkt_len);
      atomicMin(&slot->min_pkt_len, pkt_len);
      atomicMax(&slot->max_pkt_len, pkt_len);
      atomicMin((unsigned long long*)&slot->first_seen_ts,
                (unsigned long long)ts);
      atomicMax((unsigned long long*)&slot->last_seen_ts,
                (unsigned long long)ts);
      return true;
    }
    idx = (idx + 1) & HASH_MASK;  // genuinely different flow: probe on
  }
  return false;  // table full
}

__device__ bool ParseIP(const uint8_t* pkt, uint32_t caplen, const uint8_t* l3,
                        flow_key_t* key) {
  const uint8_t* end = pkt + caplen;
  if (l3 >= end) return false;

  uint8_t version = l3[0] >> 4;
  const uint8_t* l4;
  bool ports_ok = true;  // false for non-first fragments: no L4 header there
  uint8_t proto;

  if (version == 4) {
    if (end - l3 < 20) return false;
    uint32_t ihl = (uint32_t)(l3[0] & 0x0F) * 4;
    if (ihl < 20 || (uint32_t)(end - l3) < ihl) return false;

    key->ip_ver = 4;
    key->src_ip[0] = be32(l3 + 12);
    key->dst_ip[0] = be32(l3 + 16);
    for (int i = 1; i < 4; i++) key->src_ip[i] = key->dst_ip[i] = 0;
    proto = l3[9];
    ports_ok = (be16(l3 + 6) & 0x1FFF) == 0;  // fragment offset
    l4 = l3 + ihl;
  } else if (version == 6) {
    if (end - l3 < 40) return false;

    key->ip_ver = 6;
    for (int i = 0; i < 4; i++) {
      key->src_ip[i] = be32(l3 + 8 + 4 * i);
      key->dst_ip[i] = be32(l3 + 24 + 4 * i);
    }

    // Walk the extension header chain (bounded) to find the real protocol.
    uint8_t nh = l3[6];
    const uint8_t* cur = l3 + 40;
    for (int n = 0; n < 8; n++) {
      uint32_t hlen;
      if (nh == 0 || nh == 43 || nh == 60) {  // hop-by-hop, routing, dest opts
        if (end - cur < 8) return false;
        hlen = ((uint32_t)cur[1] + 1) * 8;
      } else if (nh == 51) {  // authentication header
        if (end - cur < 8) return false;
        hlen = ((uint32_t)cur[1] + 2) * 4;
      } else if (nh == 44) {  // fragment header
        if (end - cur < 8) return false;
        if ((be16(cur + 2) >> 3) != 0) ports_ok = false;
        hlen = 8;
      } else {
        break;
      }
      if ((uint32_t)(end - cur) < hlen) return false;
      nh = cur[0];
      cur += hlen;
    }
    proto = nh;
    l4 = cur;
  } else {
    return false;
  }

  key->proto = proto;
  key->src_port = 0;
  key->dst_port = 0;
  // Only TCP/UDP/SCTP have ports. For ICMP & co. the bytes there are type /
  // checksum, which would split one logical flow into many.
  if (ports_ok && (proto == TCP || proto == UDP || proto == SCTP) &&
      end - l4 >= 4) {
    key->src_port = be16(l4);
    key->dst_port = be16(l4 + 2);
  }
  return true;
}

__global__ void PcapParallelParsing(const uint8_t* d_packet_data,
                                    const pkt_info_t* d_packets,
                                    size_t d_total_packets,
                                    flow_record_t* d_flow_table,
                                    parse_stats_t* d_stats) {
  size_t th_idx = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
  if (th_idx >= d_total_packets) return;

  const pkt_info_t info = d_packets[th_idx];
  const uint8_t* pkt = d_packet_data + info.offset;
  const uint8_t* end = pkt + info.caplen;

  if (info.caplen < 14) {
    atomicAdd(&d_stats->malformed, 1ULL);
    return;
  }

  uint16_t ethertype = be16(pkt + 12);
  const uint8_t* cur = pkt + 14;
  bool malformed = false;

  // VLAN tags (802.1Q / 802.1ad), possibly stacked
  for (int d = 0; d < 8 && (ethertype == VLAN_TAG || ethertype == QINQ_TAG);
       d++) {
    if (end - cur < 4) {
      malformed = true;
      break;
    }
    ethertype = be16(cur + 2);
    cur += 4;
  }

  const uint8_t* l3 = NULL;
  if (!malformed) {
    switch (ethertype) {
      case IPv4:
      case IPv6:
        l3 = cur;
        break;
      case MLPS_UNI:
      case MLPS_MULTI: {
        // pop labels until the bottom-of-stack bit (bounded)
        bool bos = false;
        for (int d = 0; d < 16 && !bos; d++) {
          if (end - cur < 4) {
            malformed = true;
            break;
          }
          bos = (cur[2] & 0x01) != 0;
          cur += 4;
        }
        if (!malformed) {
          if (!bos) {
            malformed = true;
          } else if (end - cur >= 1 &&
                     ((cur[0] >> 4) == 4 || (cur[0] >> 4) == 6)) {
            l3 = cur;
          }
        }
        break;
      }
      case PPPoE_SESSION: {
        // 6-byte PPPoE header + 2-byte PPP protocol: 0x0021 = IPv4, 0x0057 = IPv6
        if (end - cur < 8) {
          malformed = true;
          break;
        }
        uint16_t ppp = be16(cur + 6);
        if (ppp == 0x0021 || ppp == 0x0057) l3 = cur + 8;
        break;
      }
      default:  // ARP, PPPoE discovery, anything else: no flow
        break;
    }
  }

  if (malformed) {
    atomicAdd(&d_stats->malformed, 1ULL);
    return;
  }
  if (l3 == NULL) {
    atomicAdd(&d_stats->non_flow, 1ULL);
    return;
  }

  flow_key_t key;
  key.file_id = info.file_id;
  if (!ParseIP(pkt, info.caplen, l3, &key)) {
    atomicAdd(&d_stats->malformed, 1ULL);
    return;
  }

  // Length = frame length on the wire (pcap orig_len), independent of the IP
  // header, so TSO/GSO captures (IP total length 0) and padding are handled.
  if (!AggregateFlow(d_flow_table, key, info.wire_len, info.ts_ns)) {
    atomicAdd(&d_stats->table_full, 1ULL);
  }
}


// Needs a local variable `my_rank` and `rc`, and a `cleanup:` label.
#define CUDA_CHECK(call)                                                  \
  do {                                                                    \
    cudaError_t err_ = (call);                                            \
    if (err_ != cudaSuccess) {                                            \
      fprintf(stderr, "[Rank %d] CUDA error at %s:%d: %s\n", my_rank,     \
              __FILE__, __LINE__, cudaGetErrorString(err_));              \
      rc = 1;                                                             \
      goto cleanup;                                                       \
    }                                                                     \
  } while (0)

static void FormatIP(const flow_key_t* k, const uint32_t* a, char* buf,
                     size_t n) {
  if (k->ip_ver == 4) {
    snprintf(buf, n, "%u.%u.%u.%u", a[0] >> 24, (a[0] >> 16) & 0xFF,
             (a[0] >> 8) & 0xFF, a[0] & 0xFF);
  } else {
    snprintf(buf, n, "%x:%x:%x:%x:%x:%x:%x:%x", a[0] >> 16, a[0] & 0xFFFF,
             a[1] >> 16, a[1] & 0xFFFF, a[2] >> 16, a[2] & 0xFFFF, a[3] >> 16,
             a[3] & 0xFFFF);
  }
}

// Does all the per-rank work. Returns 0 on success. Every exit goes through
// `cleanup`, so main() always reaches MPI_Finalize.
static int RunRank(int my_rank, int num_procs, int threads_per_block) {
  // All declarations up front: a goto may not jump over an initialisation.
  int rc = 0;
  int local_rank = 0, dev_count = 0;
  MPI_Comm local_comm;
  uint8_t* d_packet_data = NULL;
  pkt_info_t* d_packets = NULL;
  flow_record_t* d_flow_table = NULL;
  parse_stats_t* d_stats = NULL;
  flow_record_t* h_flow_table = NULL;
  parse_stats_t h_stats;
  size_t cuda_block_count = 0;
  size_t active_flows = 0, packets_in_flows = 0, bad_records = 0;
  size_t accounted = 0;
  const size_t table_bytes = sizeof(flow_record_t) * (size_t)HASH_TABLE_SIZE;

  // One GPU per rank on its node (collective: every rank must reach this).
  MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, my_rank,
                      MPI_INFO_NULL, &local_comm);
  MPI_Comm_rank(local_comm, &local_rank);
  MPI_Comm_free(&local_comm);

  CUDA_CHECK(cudaGetDeviceCount(&dev_count));
  if (dev_count == 0) {
    fprintf(stderr, "[Rank %d] no CUDA device found\n", my_rank);
    rc = 1;
    goto cleanup;
  }
  CUDA_CHECK(cudaSetDevice(local_rank % dev_count));

  // Each rank parses its own subset of files into its local buffers
  if (DatasetRead(my_rank, num_procs) != 0) {
    rc = 1;
    goto cleanup;
  }
  printf("[Rank %d] Finished loading. Total packets on this rank: %zu\n",
         my_rank, total_packets);

  if (total_packets == 0) {
    printf("[Rank %d] nothing to process\n", my_rank);
    goto cleanup;
  }

  CUDA_CHECK(cudaMalloc(&d_packet_data, total_data_bytes));
  CUDA_CHECK(cudaMalloc(&d_packets, sizeof(pkt_info_t) * total_packets));
  CUDA_CHECK(cudaMalloc(&d_flow_table, table_bytes));
  CUDA_CHECK(cudaMalloc(&d_stats, sizeof(parse_stats_t)));

  CUDA_CHECK(cudaMemcpy(d_packet_data, packet_data, total_data_bytes,
                        cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_packets, packets,
                        sizeof(pkt_info_t) * total_packets,
                        cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemset(d_flow_table, 0, table_bytes));
  CUDA_CHECK(cudaMemset(d_stats, 0, sizeof(parse_stats_t)));

  cuda_block_count = (total_packets + threads_per_block - 1) / threads_per_block;
  PcapParallelParsing<<<(unsigned int)cuda_block_count, threads_per_block>>>(
      d_packet_data, d_packets, total_packets, d_flow_table, d_stats);
  CUDA_CHECK(cudaGetLastError());       // launch errors
  CUDA_CHECK(cudaDeviceSynchronize());  // execution errors

  h_flow_table = (flow_record_t*)malloc(table_bytes);
  if (h_flow_table == NULL) {
    fprintf(stderr, "[Rank %d] out of host memory for the flow table\n",
            my_rank);
    rc = 1;
    goto cleanup;
  }
  CUDA_CHECK(cudaMemcpy(h_flow_table, d_flow_table, table_bytes,
                        cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(&h_stats, d_stats, sizeof(parse_stats_t),
                        cudaMemcpyDeviceToHost));

  for (size_t i = 0; i < (size_t)HASH_TABLE_SIZE; i++) {
    const flow_record_t* f = &h_flow_table[i];
    if (f->state != SLOT_READY) continue;

    if (f->packet_count == 0 || f->min_pkt_len > f->max_pkt_len ||
        f->first_seen_ts > f->last_seen_ts) {
      bad_records++;
    }

    if (active_flows < 5) {
      char src[48], dst[48];
      FormatIP(&f->key, f->key.src_ip, src, sizeof(src));
      FormatIP(&f->key, f->key.dst_ip, dst, sizeof(dst));
      printf(
          "[Rank %d] flow file=%u %s:%u -> %s:%u proto=%u pkts=%u bytes=%llu"
          " len[min=%u max=%u] first=%llu last=%llu\n",
          my_rank, f->key.file_id, src, f->key.src_port, dst, f->key.dst_port,
          f->key.proto, f->packet_count, (unsigned long long)f->byte_count,
          f->min_pkt_len, f->max_pkt_len, (unsigned long long)f->first_seen_ts,
          (unsigned long long)f->last_seen_ts);
    }
    packets_in_flows += f->packet_count;
    active_flows++;
  }

  accounted = packets_in_flows + h_stats.non_flow + h_stats.malformed +
              h_stats.table_full;
  printf("[Rank %d] Total unique flows detected: %zu\n", my_rank, active_flows);
  printf(
      "[Rank %d] packets: total=%zu in_flows=%zu non_flow=%llu malformed=%llu"
      " table_full=%llu -> %s\n",
      my_rank, total_packets, packets_in_flows, h_stats.non_flow,
      h_stats.malformed, h_stats.table_full,
      accounted == total_packets ? "all accounted for" : "MISMATCH");
  if (h_stats.table_full > 0) {
    fprintf(stderr,
            "[Rank %d] WARNING: flow table full, %llu packets were dropped\n",
            my_rank, h_stats.table_full);
  }
  if (bad_records > 0) {
    fprintf(stderr, "[Rank %d] WARNING: %zu flow records failed sanity checks\n",
            my_rank, bad_records);
  }

cleanup:
  cudaFree(d_packet_data);
  cudaFree(d_packets);
  cudaFree(d_flow_table);
  cudaFree(d_stats);
  free(h_flow_table);
  free(packet_data);
  free(packets);
  packet_data = NULL;
  packets = NULL;
  return rc;
}

// arguments: ./executable <CUDA_THREAD_NUMBER>
int main(int argc, char* argv[]) {
  MPI_Init(&argc, &argv);

  int my_rank, num_procs;
  MPI_Comm_rank(MPI_COMM_WORLD, &my_rank);
  MPI_Comm_size(MPI_COMM_WORLD, &num_procs);

  int threads_per_block = 1024;
  if (argc >= 2) {
    int t = atoi(argv[1]);
    if (t >= 1 && t <= 1024) {
      threads_per_block = t;
    } else if (my_rank == 0) {
      fprintf(stderr, "Ignoring invalid thread count '%s', using 1024\n",
              argv[1]);
    }
  }

  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);
  int rc = RunRank(my_rank, num_procs, threads_per_block);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  float milliseconds = 0.0f;
  cudaEventElapsedTime(&milliseconds, start, stop);
  float seconds = milliseconds/1000;

  printf("[RANK %d]: Took %.12f seconds to execute pipeline\n", my_rank, seconds);

  cudaEventDestroy(start);
  cudaEventDestroy(stop);
  MPI_Finalize();
  return rc;
}


Overwriting gpu.cu


## GPU Output

Compile the code

In [25]:
!nvcc -ccbin mpicxx gpu.cu -o gpu

Run the code

In [26]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./gpu

[Rank 0] file_id=0 6to4.pcap (5 packets)
/content/drive/MyDrive/Colab Notebooks/dataset/Apple_IP-over-IEEE_1394_Packet.pcap: link type 138 is not Ethernet, skipped
[Rank 2] file_id=2 DICOM_C-ECHO-echoscu.pcap (25 packets)
[Rank 3] file_id=3 IGMP-dataset.pcap (147 packets)
[Rank 0] file_id=4 Mobile-Originating-Call(AMR).pcap (299 packets)
/content/drive/MyDrive/Colab Notebooks/dataset/Network_Join_Nokia_Mobile.pcap: link type 105 is not Ethernet, skipped
[Rank 1] file_id=5 Mobile-Terminating-Call(AMR).pcap (308 packets)
[Rank 2] file_id=6 NTLM-wenchao.pcap (80 packets)
[Rank 0] file_id=8 ST2110-40-OP47_Teletext.pcap (1336 packets)
[Rank 3] file_id=11 TNS_Oracle2.pcap (36 packets)
[Rank 2] file_id=10 TNS_Oracle1.pcap (78 packets)
[Rank 1] file_id=9 ST2110-40_ancillary_data.pcap (1000 packets)
[Rank 0] file_id=12 TNS_Oracle3.pcap (36 packets)
[Rank 3] file_id=15 Teredo.pcap (78 packets)
[Rank 1] file_id=13 TNS_Oracle4.pcap (10 packets)
[Rank 2] file_id=14 TNS_Oracle5.pcap (56 packets)
[Ra

# CPU Baseline

In [1]:
%%file cpu.c
#include <dirent.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <inttypes.h>
#include <stdbool.h>
#include <strings.h>
#include <sys/stat.h>
#include <time.h>

/* Batching defines */
#define DATASET_PATH "/content/drive/MyDrive/Colab Notebooks/dataset"

// A single captured packet larger than this means a corrupt record.
#define MAX_PKT_BYTES (1u << 20)

/* Parsing defines */
// ethernet types
#define IPv4 0x0800
#define ARP 0x0806
#define IPv6 0x86DD
#define VLAN_TAG 0x8100
#define QINQ_TAG 0x88A8
#define PPPoE_DISCOV 0x8863
#define PPPoE_SESSION 0x8864
#define MLPS_UNI 0x8847
#define MLPS_MULTI 0x8848

// IP protocol types
#define ICMP 1
#define TCP 6
#define UDP 17
#define SCTP 132

// arp operations
#define ARP_REQ 1
#define ARP_REP 2

// Hashing
#define HASH_TABLE_SIZE (1 << 20)
#define HASH_MASK (HASH_TABLE_SIZE - 1)
#define SLOT_EMPTY 0
#define SLOT_WRITING 1
#define SLOT_READY 2

/* Batching structs */
typedef struct {
  uint32_t magic_number;   // number identifying PCAP format
  uint16_t version_major;  // Major version number
  uint16_t version_minor;  // Minor version number
  int32_t thiszone;        // GMT to local correction
  uint32_t sigfigs;        // Accuracy of timestamps
  uint32_t snaplen;        // Max length of captured packets
  uint32_t network;        // Data link type
} __attribute__((packed)) pcap_hdr_t;

// PCAP packet header
typedef struct {
  uint32_t ts_sec;    // Timestamp seconds
  uint32_t ts_usec;   // Timestamp microseconds (nanoseconds for 0xa1b23c4d)
  uint32_t incl_len;  // Number of bytes of packet saved in file
  uint32_t orig_len;  // Actual length of packet on the wire
} __attribute__((packed)) pcap_pkt_hdr_t;

// Per-packet metadata, one 32-byte record per packet (shared by host + GPU).
typedef struct {
  uint64_t offset;    // byte offset of the packet inside packet_data
  uint64_t ts_ns;     // capture timestamp, nanoseconds
  uint32_t caplen;    // bytes actually stored in packet_data (<= wire_len)
  uint32_t wire_len;  // length of the frame on the wire
  uint32_t file_id;   // index of the source pcap (keeps captures separate)
  uint32_t _pad;
} pkt_info_t;

/* Parsing structs */
typedef struct {
  uint32_t file_id;
  uint32_t src_ip[4];
  uint32_t dst_ip[4];
  uint16_t src_port;  // 0 unless TCP/UDP/SCTP
  uint16_t dst_port;
  uint8_t proto;   // final upper-layer protocol (after IPv6 ext. headers)
  uint8_t ip_ver;  // 4 or 6
} flow_key_t;

typedef struct {
  uint32_t state;
  flow_key_t key;  // written once by the slot owner
  // aggregates (atomics)
  uint32_t packet_count;
  uint32_t min_pkt_len;
  uint32_t max_pkt_len;
  uint64_t byte_count;
  uint64_t first_seen_ts;  // ns
  uint64_t last_seen_ts;   // ns
} flow_record_t;

// Every packet lands in exactly one bucket, so for each rank:
//   sum(packet_count) + non_flow + malformed + table_full == total_packets
typedef struct {
  unsigned long long non_flow;    // ARP, PPPoE discovery, unknown ethertype...
  unsigned long long malformed;   // truncated or invalid headers
  unsigned long long table_full;  // no free slot in the hash table
} parse_stats_t;


/* Batching functions*/

uint8_t* packet_data = NULL;
pkt_info_t* packets = NULL;
size_t total_data_bytes = 0;
size_t total_packets = 0;
size_t data_capacity = 0;
size_t packet_capacity = 0;

static uint32_t Swap32(uint32_t v) { return __builtin_bswap32(v); }

// Grow both buffers geometrically. Returns 0 on success, -1 if out of memory.
static int EnsureCapacity(size_t need_packets, size_t need_bytes) {
  if (packets == NULL || need_packets > packet_capacity) {
    size_t cap = packet_capacity ? packet_capacity : 1024;
    while (cap < need_packets) cap *= 2;
    pkt_info_t* p = (pkt_info_t*)realloc(packets, cap * sizeof(pkt_info_t));
    if (p == NULL) return -1;
    packets = p;
    packet_capacity = cap;
  }
  if (packet_data == NULL || need_bytes > data_capacity) {
    size_t cap = data_capacity ? data_capacity : 65536;
    while (cap < need_bytes) cap *= 2;
    uint8_t* p = (uint8_t*)realloc(packet_data, cap);
    if (p == NULL) return -1;
    packet_data = p;
    data_capacity = cap;
  }
  return 0;
}

int PcapRead(const char* pcap_file, uint32_t file_id) {
  FILE* fd = fopen(pcap_file, "rb");
  if (fd == NULL) {
    fprintf(stderr, "Cannot open %s: ", pcap_file);
    perror(NULL);
    return 1;
  }

  pcap_hdr_t glob_hdr;
  if (fread(&glob_hdr, sizeof(glob_hdr), 1, fd) != 1) {
    fprintf(stderr, "%s: cannot read the pcap global header, skipped\n",
            pcap_file);
    fclose(fd);
    return 1;
  }

  // Classic pcap only. The magic tells us byte order and timestamp resolution.
  int swap = 0, nano = 0;
  switch (glob_hdr.magic_number) {
    case 0xa1b2c3d4:
      swap = 0;
      nano = 0;
      break;  // native, microseconds
    case 0xa1b23c4d:
      swap = 0;
      nano = 1;
      break;  // native, nanoseconds
    case 0xd4c3b2a1:
      swap = 1;
      nano = 0;
      break;  // swapped, microseconds
    case 0x4d3cb2a1:
      swap = 1;
      nano = 1;
      break;  // swapped, nanoseconds
    default:
      fprintf(stderr,
              "%s: unsupported format (magic 0x%08x, pcapng?), skipped\n",
              pcap_file, glob_hdr.magic_number);
      fclose(fd);
      return 1;
  }

  // The parser only understands Ethernet framing (linktype 1). The top 4 bits
  // of the field carry FCS info, so mask them off.
  uint32_t linktype =
      (swap ? Swap32(glob_hdr.network) : glob_hdr.network) & 0x0FFFFFFFu;
  if (linktype != 1) {
    fprintf(stderr, "%s: link type %u is not Ethernet, skipped\n", pcap_file,
            linktype);
    fclose(fd);
    return 1;
  }

  pcap_pkt_hdr_t pkt_hdr;
  while (fread(&pkt_hdr, sizeof(pkt_hdr), 1, fd) == 1) {
    uint32_t ts_sec = pkt_hdr.ts_sec;
    uint32_t ts_frac = pkt_hdr.ts_usec;
    uint32_t caplen = pkt_hdr.incl_len;
    uint32_t wirelen = pkt_hdr.orig_len;
    if (swap) {
      ts_sec = Swap32(ts_sec);
      ts_frac = Swap32(ts_frac);
      caplen = Swap32(caplen);
      wirelen = Swap32(wirelen);
    }

    if (caplen > MAX_PKT_BYTES) {
      fprintf(stderr,
              "%s: corrupt record (incl_len=%u), rest of file ignored\n",
              pcap_file, caplen);
      break;
    }

    if (EnsureCapacity(total_packets + 1, total_data_bytes + caplen) != 0) {
      fprintf(stderr, "Out of memory while loading %s\n", pcap_file);
      fclose(fd);
      return -1;
    }

    // Read the payload first; a cut-off last packet is dropped, not recorded.
    if (caplen > 0 &&
        fread(packet_data + total_data_bytes, 1, caplen, fd) != caplen) {
      fprintf(stderr, "%s: truncated final packet dropped\n", pcap_file);
      break;
    }

    pkt_info_t* p = &packets[total_packets];
    p->offset = total_data_bytes;
    p->ts_ns = (uint64_t)ts_sec * 1000000000ULL +
               (nano ? (uint64_t)ts_frac : (uint64_t)ts_frac * 1000ULL);
    p->caplen = caplen;
    p->wire_len = wirelen < caplen ? caplen : wirelen;  // guard bad orig_len
    p->file_id = file_id;
    p->_pad = 0;

    total_data_bytes += caplen;
    total_packets++;
  }

  fclose(fd);
  return 0;
}

static int HasSuffix(const char* s, const char* suffix) {
  size_t n = strlen(s), m = strlen(suffix);
  return n >= m && strcasecmp(s + n - m, suffix) == 0;
}

int DatasetRead() {
  struct dirent** list = NULL;
  // scandir + alphasort gives every rank the same file order, so the
  // round-robin split below never skips or duplicates a file.
  int n = scandir(DATASET_PATH, &list, NULL, alphasort);
  if (n < 0) {
    perror("Error opening dataset directory");
    return 1;
  }

  char filepath[1024];
  uint32_t file_id = 0;  // counts candidate files
  int loaded = 0, skipped = 0, rc = 0;

  for (int i = 0; i < n && rc == 0; i++) {
    const char* name = list[i]->d_name;
    if (!HasSuffix(name, ".pcap") && !HasSuffix(name, ".cap")) continue;

    int len = snprintf(filepath, sizeof(filepath), "%s/%s", DATASET_PATH, name);
    if (len < 0 || (size_t)len >= sizeof(filepath)) continue;

    // d_type can be DT_UNKNOWN on some filesystems, so ask stat() instead
    struct stat st;
    if (stat(filepath, &st) != 0 || !S_ISREG(st.st_mode)) continue;

    uint32_t this_id = file_id++;

    size_t before = total_packets;
    int r = PcapRead(filepath, this_id);
    if (r < 0) {
      rc = 1;
    } else if (r > 0) {
      skipped++;
    } else {
      loaded++;

      printf("file_id=%u %s (%zu packets)\n", this_id, name,
             total_packets - before);
    }
  }

  for (int i = 0; i < n; i++) free(list[i]);
  free(list);

  printf("files loaded: %d, skipped: %d\n", loaded, skipped);
  return rc;
}


/*Processing functions*/

static inline uint16_t be16(const uint8_t* p) {
  return ((uint16_t)p[0] << 8) | p[1];
}

static inline uint32_t be32(const uint8_t* p) {
  return ((uint32_t)p[0] << 24) | ((uint32_t)p[1] << 16) |
         ((uint32_t)p[2] << 8) | (uint32_t)p[3];
}

static inline uint32_t Mix(uint32_t h, uint32_t x) {
  h ^= x;
  return h * 16777619u;  // FNV prime
}

uint32_t HashKey(const flow_key_t* k) {
  uint32_t h = 2166136261u;  // FNV offset basis
  h = Mix(h, k->file_id);

  for (int i = 0; i < 4; i++) h = Mix(h, k->src_ip[i]);

  for (int i = 0; i < 4; i++) h = Mix(h, k->dst_ip[i]);
  h = Mix(h, ((uint32_t)k->src_port << 16) | k->dst_port);
  h = Mix(h, ((uint32_t)k->proto << 8) | k->ip_ver);
  // murmur3 finalizer: a multiply only carries information upward, so without
  // this the bits kept by HASH_MASK ignore most of the high input bits (e.g.
  // the upper bits of the source port).
  h ^= h >> 16;
  h *= 0x85ebca6bu;
  h ^= h >> 13;
  h *= 0xc2b2ae35u;
  h ^= h >> 16;
  return h;
}

static inline void StoreKey(volatile flow_key_t* d, const flow_key_t* k) {
  d->file_id = k->file_id;
  for (int i = 0; i < 4; i++) {
    d->src_ip[i] = k->src_ip[i];
    d->dst_ip[i] = k->dst_ip[i];
  }
  d->src_port = k->src_port;
  d->dst_port = k->dst_port;
  d->proto = k->proto;
  d->ip_ver = k->ip_ver;
}

static inline void LoadKey(const volatile flow_key_t* s, flow_key_t* k) {
  k->file_id = s->file_id;
  for (int i = 0; i < 4; i++) {
    k->src_ip[i] = s->src_ip[i];
    k->dst_ip[i] = s->dst_ip[i];
  }
  k->src_port = s->src_port;
  k->dst_port = s->dst_port;
  k->proto = s->proto;
  k->ip_ver = s->ip_ver;
}

static inline bool KeyEq(const flow_key_t* a, const flow_key_t* b) {
  if (a->file_id != b->file_id || a->src_port != b->src_port ||
      a->dst_port != b->dst_port || a->proto != b->proto ||
      a->ip_ver != b->ip_ver)
    return false;
#pragma unroll
  for (int i = 0; i < 4; i++) {
    if (a->src_ip[i] != b->src_ip[i] || a->dst_ip[i] != b->dst_ip[i])
      return false;
  }
  return true;
}

bool AggregateFlow(flow_record_t* table, const flow_key_t* key,
                   uint32_t pkt_len, uint64_t ts) {
  uint32_t idx = HashKey(key) & HASH_MASK;

  for (uint32_t probes = 0; probes < HASH_TABLE_SIZE; probes++) {
    flow_record_t* slot = &table[idx];

    // Since this table is private to the thread (or protected),
    // we check if the slot is empty (state == SLOT_EMPTY or src_ip == 0)
    if (slot->state == SLOT_EMPTY) {
      slot->state = SLOT_READY;
      slot->key = *key;  // Assuming direct assignment or memcpy for flow_key_t

      // Initialize boundaries
      slot->min_pkt_len = UINT32_MAX;
      slot->first_seen_ts = UINT64_MAX;
      slot->packet_count = 0;
      slot->byte_count = 0;
      slot->max_pkt_len = 0;
      slot->last_seen_ts = 0;
    }

    // Does the key match this slot?
    // (Assuming KeyEq can be evaluated directly on CPU structs)
    if (KeyEq(&(slot->key), key)) {
      slot->packet_count++;
      slot->byte_count += pkt_len;

      if (pkt_len < slot->min_pkt_len) slot->min_pkt_len = pkt_len;
      if (pkt_len > slot->max_pkt_len) slot->max_pkt_len = pkt_len;
      if (ts < slot->first_seen_ts) slot->first_seen_ts = ts;
      if (ts > slot->last_seen_ts) slot->last_seen_ts = ts;

      return true;
    }

    idx = (idx + 1) & HASH_MASK;  // Linear probe
  }
  return false;  // Table full
}

bool ParseIP(const uint8_t* pkt, uint32_t caplen, const uint8_t* l3,
             flow_key_t* key) {
  const uint8_t* end = pkt + caplen;
  if (l3 >= end) return false;

  uint8_t version = l3[0] >> 4;
  const uint8_t* l4;
  bool ports_ok = true;  // false for non-first fragments: no L4 header there
  uint8_t proto;

  if (version == 4) {
    if (end - l3 < 20) return false;
    uint32_t ihl = (uint32_t)(l3[0] & 0x0F) * 4;
    if (ihl < 20 || (uint32_t)(end - l3) < ihl) return false;

    key->ip_ver = 4;
    key->src_ip[0] = be32(l3 + 12);
    key->dst_ip[0] = be32(l3 + 16);
    for (int i = 1; i < 4; i++) key->src_ip[i] = key->dst_ip[i] = 0;
    proto = l3[9];
    ports_ok = (be16(l3 + 6) & 0x1FFF) == 0;  // fragment offset
    l4 = l3 + ihl;
  } else if (version == 6) {
    if (end - l3 < 40) return false;

    key->ip_ver = 6;
    for (int i = 0; i < 4; i++) {
      key->src_ip[i] = be32(l3 + 8 + 4 * i);
      key->dst_ip[i] = be32(l3 + 24 + 4 * i);
    }

    // Walk the extension header chain (bounded) to find the real protocol.
    uint8_t nh = l3[6];
    const uint8_t* cur = l3 + 40;
    for (int n = 0; n < 8; n++) {
      uint32_t hlen;
      if (nh == 0 || nh == 43 || nh == 60) {  // hop-by-hop, routing, dest opts
        if (end - cur < 8) return false;
        hlen = ((uint32_t)cur[1] + 1) * 8;
      } else if (nh == 51) {  // authentication header
        if (end - cur < 8) return false;
        hlen = ((uint32_t)cur[1] + 2) * 4;
      } else if (nh == 44) {  // fragment header
        if (end - cur < 8) return false;
        if ((be16(cur + 2) >> 3) != 0) ports_ok = false;
        hlen = 8;
      } else {
        break;
      }
      if ((uint32_t)(end - cur) < hlen) return false;
      nh = cur[0];
      cur += hlen;
    }
    proto = nh;
    l4 = cur;
  } else {
    return false;
  }

  key->proto = proto;
  key->src_port = 0;
  key->dst_port = 0;
  // Only TCP/UDP/SCTP have ports. For ICMP & co. the bytes there are type /
  // checksum, which would split one logical flow into many.
  if (ports_ok && (proto == TCP || proto == UDP || proto == SCTP) &&
      end - l4 >= 4) {
    key->src_port = be16(l4);
    key->dst_port = be16(l4 + 2);
  }
  return true;
}

void ProcessPacketsOpenMP(const uint8_t* packet_data, const pkt_info_t* packets,
                          size_t total_packets, flow_record_t* global_table,
                          parse_stats_t* stats) {
  // Initialize global stats
  stats->malformed = 0;
  stats->non_flow = 0;
  stats->table_full = 0;

#pragma omp parallel
  {
    // 1. Allocate a thread-private hash table for zero-contention packet
    // processing
    flow_record_t* local_table =
        (flow_record_t*)calloc(HASH_TABLE_SIZE, sizeof(flow_record_t));

    // 2. Thread-local statistics counters to avoid atomic overhead in the hot
    // loop
    parse_stats_t local_stats = {0, 0, 0};

// 3. Parallelize the packet iteration loop dynamically
#pragma omp for schedule(dynamic, 256)
    for (size_t i = 0; i < total_packets; i++) {
      const pkt_info_t info = packets[i];
      const uint8_t* pkt = packet_data + info.offset;
      const uint8_t* end = pkt + info.caplen;

      if (info.caplen < 14) {
        local_stats.malformed++;
        continue;
      }

      uint16_t ethertype = be16(pkt + 12);
      const uint8_t* cur = pkt + 14;
      bool malformed = false;

      // VLAN tags (802.1Q / 802.1ad), possibly stacked
      for (int d = 0; d < 8 && (ethertype == VLAN_TAG || ethertype == QINQ_TAG);
           d++) {
        if (end - cur < 4) {
          malformed = true;
          break;
        }
        ethertype = be16(cur + 2);
        cur += 4;
      }

      const uint8_t* l3 = NULL;
      if (!malformed) {
        switch (ethertype) {
          case IPv4:
          case IPv6:
            l3 = cur;
            break;
          case MLPS_UNI:
          case MLPS_MULTI: {
            bool bos = false;
            for (int d = 0; d < 16 && !bos; d++) {
              if (end - cur < 4) {
                malformed = true;
                break;
              }
              bos = (cur[2] & 0x01) != 0;
              cur += 4;
            }
            if (!malformed) {
              if (!bos) {
                malformed = true;
              } else if (end - cur >= 1 &&
                         ((cur[0] >> 4) == 4 || (cur[0] >> 4) == 6)) {
                l3 = cur;
              }
            }
            break;
          }
          case PPPoE_SESSION: {
            if (end - cur < 8) {
              malformed = true;
              break;
            }
            uint16_t ppp = be16(cur + 6);
            if (ppp == 0x0021 || ppp == 0x0057) l3 = cur + 8;
            break;
          }
          default:
            break;
        }
      }

      if (malformed) {
        local_stats.malformed++;
        continue;
      }
      if (l3 == NULL) {
        local_stats.non_flow++;
        continue;
      }

      flow_key_t key;
      key.file_id = info.file_id;
      if (!ParseIP(pkt, info.caplen, l3, &key)) {
        local_stats.malformed++;
        continue;
      }

      // Aggregate into the thread-private hash table
      if (!AggregateFlow(local_table, &key, info.wire_len, info.ts_ns)) {
        local_stats.table_full++;
      }
    }

// 4. Safely merge thread-local tables and stats into the global structures at
// the end
#pragma omp critical
    {
      stats->malformed += local_stats.malformed;
      stats->non_flow += local_stats.non_flow;
      stats->table_full += local_stats.table_full;

      // Merge each populated slot from local_table into global_table
      for (uint32_t i = 0; i < HASH_TABLE_SIZE; i++) {
        if (local_table[i].state == SLOT_READY) {
          flow_record_t local_rec = local_table[i];
          uint32_t idx = HashKey(&(local_rec.key)) & HASH_MASK;

          // Linear probe into the global table to merge counts and metrics
          for (uint32_t probes = 0; probes < HASH_TABLE_SIZE; probes++) {
            flow_record_t* g_slot = &global_table[idx];

            if (g_slot->state == SLOT_EMPTY) {
              g_slot->state = SLOT_READY;
              g_slot->key = local_rec.key;
              g_slot->packet_count = local_rec.packet_count;
              g_slot->byte_count = local_rec.byte_count;
              g_slot->min_pkt_len = local_rec.min_pkt_len;
              g_slot->max_pkt_len = local_rec.max_pkt_len;
              g_slot->first_seen_ts = local_rec.first_seen_ts;
              g_slot->last_seen_ts = local_rec.last_seen_ts;
              break;
            }

            if (KeyEq(&(g_slot->key), &(local_rec.key))) {
              g_slot->packet_count += local_rec.packet_count;
              g_slot->byte_count += local_rec.byte_count;
              if (local_rec.min_pkt_len < g_slot->min_pkt_len)
                g_slot->min_pkt_len = local_rec.min_pkt_len;
              if (local_rec.max_pkt_len > g_slot->max_pkt_len)
                g_slot->max_pkt_len = local_rec.max_pkt_len;
              if (local_rec.first_seen_ts < g_slot->first_seen_ts)
                g_slot->first_seen_ts = local_rec.first_seen_ts;
              if (local_rec.last_seen_ts > g_slot->last_seen_ts)
                g_slot->last_seen_ts = local_rec.last_seen_ts;
              break;
            }

            idx = (idx + 1) & HASH_MASK;
          }
        }
      }
    }

    free(local_table);
  }
}

/* main */

// Helper function to format IPv4 and IPv6 addresses for printing
static void FormatIP(const flow_key_t* k, const uint32_t* a, char* buf,
                     size_t n) {
  if (k->ip_ver == 4) {
    snprintf(buf, n, "%u.%u.%u.%u", a[0] >> 24, (a[0] >> 16) & 0xFF,
             (a[0] >> 8) & 0xFF, a[0] & 0xFF);
  } else {
    snprintf(buf, n, "%x:%x:%x:%x:%x:%x:%x:%x", a[0] >> 16, a[0] & 0xFFFF,
             a[1] >> 16, a[1] & 0xFFFF, a[2] >> 16, a[2] & 0xFFFF, a[3] >> 16,
             a[3] & 0xFFFF);
  }
}

// ./cpu <number of OMP threads>
int main(int argc, char* argv[]) {
  int thread_count = 2;

  if (argc > 1) {
    int t = atoi(argv[1]);
    if (t >= 1) {
      thread_count = t;
    } else {
      fprintf(stderr, "Ignoring invalid thread count '%s', using 2\n", argv[1]);
    }
  }

  omp_set_num_threads(thread_count);
  printf("[CPU-OMP] Running with %d OpenMP threads\n", thread_count);

  struct timespec chron_s, chron_e;
  clock_gettime(CLOCK_MONOTONIC, &chron_s);

  // Read dataset files sequentially into flat host arrays
  if (DatasetRead() != 0) {
    fprintf(stderr, "Error reading dataset directory or pcap files.\n");
    return 1;
  }
  printf("Finished loading. Total packets loaded: %zu\n", total_packets);

  if (total_packets == 0) {
    printf("Nothing to process.\n");
    return 0;
  }

  // Allocate the global host-side flow table and stats structure
  const size_t table_bytes = sizeof(flow_record_t) * (size_t)HASH_TABLE_SIZE;
  flow_record_t* global_flow_table = (flow_record_t*)calloc(1, table_bytes);
  if (global_flow_table == NULL) {
    fprintf(stderr, "Out of host memory for the global flow table\n");
    free(packet_data);
    free(packets);
    return 1;
  }

  parse_stats_t stats = {0, 0, 0};

  // Execute the OpenMP parallel packet parsing and flow aggregation
  ProcessPacketsOpenMP(packet_data, packets, total_packets, global_flow_table,
                       &stats);

  // Post-process, validate, and print results
  size_t active_flows = 0;
  size_t packets_in_flows = 0;
  size_t bad_records = 0;

  for (size_t i = 0; i < (size_t)HASH_TABLE_SIZE; i++) {
    const flow_record_t* f = &global_flow_table[i];
    if (f->state != SLOT_READY) continue;

    if (f->packet_count == 0 || f->min_pkt_len > f->max_pkt_len ||
        f->first_seen_ts > f->last_seen_ts) {
      bad_records++;
    }

    if (active_flows < 5) {
      char src[48], dst[48];
      FormatIP(&f->key, f->key.src_ip, src, sizeof(src));
      FormatIP(&f->key, f->key.dst_ip, dst, sizeof(dst));
      printf(
          "flow file=%u %s:%u -> %s:%u proto=%u pkts=%u bytes=%llu"
          " len[min=%u max=%u] first=%llu last=%llu\n",
          f->key.file_id, src, f->key.src_port, dst, f->key.dst_port,
          f->key.proto, f->packet_count, (unsigned long long)f->byte_count,
          f->min_pkt_len, f->max_pkt_len, (unsigned long long)f->first_seen_ts,
          (unsigned long long)f->last_seen_ts);
    }
    packets_in_flows += f->packet_count;
    active_flows++;
  }

  size_t accounted =
      packets_in_flows + stats.non_flow + stats.malformed + stats.table_full;

  printf("Total unique flows detected: %zu\n", active_flows);
  printf(
      "packets: total=%zu in_flows=%zu non_flow=%llu malformed=%llu "
      "table_full=%llu -> %s\n",
      total_packets, packets_in_flows, stats.non_flow, stats.malformed,
      stats.table_full,
      accounted == total_packets ? "all accounted for" : "MISMATCH");

  if (stats.table_full > 0) {
    fprintf(stderr, "WARNING: flow table full, %llu packets were dropped\n",
            stats.table_full);
  }
  if (bad_records > 0) {
    fprintf(stderr, "WARNING: %zu flow records failed sanity checks\n",
            bad_records);
  }
  clock_gettime(CLOCK_MONOTONIC, &chron_e);
  int seconds = chron_e.tv_sec - chron_s.tv_sec;
  int nano = chron_e.tv_nsec - chron_s.tv_nsec;

  if (nano < 0) {
    seconds--;
    nano += 1000000000;
  }
  double delta = seconds + (nano / 1e9);

  printf("Took %.12f seconds to execute pipeline\n", delta);

  // Cleanup host memory allocations
  free(global_flow_table);
  free(packet_data);
  free(packets);

  return 0;
}

Writing cpu.c


Compile the code

In [3]:
!gcc cpu.c -fopenmp -o cpu

Run the executable with the number of OpenMP threads

In [8]:
!./cpu 2

[CPU-OMP] Running with 2 OpenMP threads
file_id=0 6to4.pcap (5 packets)
/content/drive/MyDrive/Colab Notebooks/dataset/Apple_IP-over-IEEE_1394_Packet.pcap: link type 138 is not Ethernet, skipped
file_id=2 DICOM_C-ECHO-echoscu.pcap (25 packets)
file_id=3 IGMP-dataset.pcap (147 packets)
file_id=4 Mobile-Originating-Call(AMR).pcap (299 packets)
file_id=5 Mobile-Terminating-Call(AMR).pcap (308 packets)
file_id=6 NTLM-wenchao.pcap (80 packets)
/content/drive/MyDrive/Colab Notebooks/dataset/Network_Join_Nokia_Mobile.pcap: link type 105 is not Ethernet, skipped
file_id=8 ST2110-40-OP47_Teletext.pcap (1336 packets)
file_id=9 ST2110-40_ancillary_data.pcap (1000 packets)
file_id=10 TNS_Oracle1.pcap (78 packets)
file_id=11 TNS_Oracle2.pcap (36 packets)
file_id=12 TNS_Oracle3.pcap (36 packets)
file_id=13 TNS_Oracle4.pcap (10 packets)
file_id=14 TNS_Oracle5.pcap (56 packets)
file_id=15 Teredo.pcap (78 packets)
file_id=16 UserLog.pcap (650 packets)
/content/drive/MyDrive/Colab Notebooks/dataset/abis